In [9]:
import torch
from torch import nn
from torchinfo import summary

In [10]:
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
x = torch.rand(size=(2,4))
net(x)

tensor([[-0.1797],
        [-0.2684]], grad_fn=<AddmmBackward0>)

In [11]:
print(net(x))  # 两行一列

tensor([[-0.1797],
        [-0.2684]], grad_fn=<AddmmBackward0>)


In [13]:
summary(net, input_size=(2,4))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [2, 1]                    --
├─Linear: 1-1                            [2, 8]                    40
├─ReLU: 1-2                              [2, 8]                    --
├─Linear: 1-3                            [2, 1]                    9
Total params: 49
Trainable params: 49
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [ ]:
print(net[2].state_dict().keys())

In [25]:
print(net.state_dict().keys())

odict_keys(['0.weight', '0.bias', '2.weight', '2.bias'])


In [15]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[ 0.3416, -0.0420, -0.3422,  0.0676, -0.1542, -0.0691,  0.3449, -0.1604]])), ('bias', tensor([-0.1373]))])


In [16]:
print(net[2].state_dict().values())

odict_values([tensor([[ 0.3416, -0.0420, -0.3422,  0.0676, -0.1542, -0.0691,  0.3449, -0.1604]]), tensor([-0.1373])])


In [17]:
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>


In [18]:
print(net[2].bias)

Parameter containing:
tensor([-0.1373], requires_grad=True)


In [19]:
print(net[2].bias.data,type(net[2].bias.data))

tensor([-0.1373]) <class 'torch.Tensor'>


In [20]:
net[2].weight.grad == None

True

In [22]:
# 一次性访问所有参数
print(*[(name,param.shape) for name,param in net.named_parameters()])
print(*[(name,param.shape) for name,param in net[0].named_parameters()])

('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))
('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))


In [24]:
print(net.state_dict()['0.bias'].data)

tensor([-0.2484,  0.2659, -0.3621, -0.3116, -0.2824, -0.1665, -0.0842, -0.1631])


In [32]:
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'第{i}个块',block1())
    return net

In [33]:
rgnet = nn.Sequential(block2(),nn.Linear(4,1))

In [34]:
rgnet(x)

tensor([[-0.2644],
        [-0.2644]], grad_fn=<AddmmBackward0>)

In [35]:
summary(rgnet, input_size=(2,4))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [2, 1]                    --
├─Sequential: 1-1                        [2, 4]                    --
│    └─Sequential: 2-1                   [2, 4]                    --
│    │    └─Linear: 3-1                  [2, 8]                    40
│    │    └─ReLU: 3-2                    [2, 8]                    --
│    │    └─Linear: 3-3                  [2, 4]                    36
│    │    └─ReLU: 3-4                    [2, 4]                    --
│    └─Sequential: 2-2                   [2, 4]                    --
│    │    └─Linear: 3-5                  [2, 8]                    40
│    │    └─ReLU: 3-6                    [2, 8]                    --
│    │    └─Linear: 3-7                  [2, 4]                    36
│    │    └─ReLU: 3-8                    [2, 4]                    --
│    └─Sequential: 2-3                   [2, 4]                    --
│    │    └─Lin

In [36]:
print(rgnet)

Sequential(
  (0): Sequential(
    (第0个块): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (第1个块): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (第2个块): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (第3个块): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [51]:
print(rgnet[0][0][0].state_dict().keys())

odict_keys(['weight', 'bias'])


In [52]:
# 参数初始化
"""
正态分布
"""
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data,net[0].bias.data

(tensor([[ 0.0145,  0.0011,  0.0148, -0.0100],
         [ 0.0060, -0.0054,  0.0018, -0.0067],
         [-0.0025, -0.0020,  0.0196, -0.0144],
         [-0.0035,  0.0003, -0.0097, -0.0023],
         [ 0.0117, -0.0078,  0.0226, -0.0108],
         [-0.0076, -0.0057, -0.0063, -0.0037],
         [ 0.0098,  0.0268,  0.0157,  0.0116],
         [ 0.0084, -0.0028, -0.0029, -0.0176]]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [54]:
"""
常数化
"""
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,2)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data,net[0].bias.data

(tensor([[2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.]]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [56]:
"""
不同层初始化
"""
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)  # 正态分布版本的 Xavier 初始化
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,42)

In [ ]:
net[0].apply(init_xavier)
net[2].apply(init_42)

In [59]:
net[0].weight.data,net[0].bias.data

(tensor([[ 0.1974, -0.6068, -0.6860,  0.4812],
         [ 0.2100, -0.2329, -0.2093,  0.3242],
         [ 0.0386,  0.0245, -0.2330,  0.1228],
         [ 0.5047,  0.1822, -0.6307,  0.1323],
         [ 0.5853,  0.5707, -0.3383,  0.3701],
         [ 0.5293, -0.0776, -0.6387,  0.6176],
         [ 0.2765,  0.6469,  0.4374, -0.4559],
         [ 0.3031,  0.5959,  0.6133,  0.7022]]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [60]:
net[2].weight.data,net[2].bias.data

(tensor([[42., 42., 42., 42., 42., 42., 42., 42.]]), tensor([0.]))

In [61]:
def my_init(m):
    if type(m) == nn.Linear:
        print('Init',*[(name,param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data *= (m.weight.data.abs() >= 5)  # 先算后面

net.apply(my_init)

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [65]:
print(net[0].weight[:2])

tensor([[-7.7622,  8.2294,  8.7049,  7.2958],
        [-7.3777, -8.7529, -0.0000,  8.9699]], grad_fn=<SliceBackward0>)


In [66]:
net[0].weight.data[:] += 1

In [74]:
net[0].weight.data[0,0] = 42
net[0].weight.data

tensor([[ 4.2000e+01,  4.1324e-01, -2.1071e-01,  3.9836e-01],
        [ 3.0274e-01, -1.1818e-01,  3.3072e-01, -7.6988e-02],
        [-2.5378e-01,  4.9417e-02,  4.6985e-01,  2.3767e-01],
        [-2.7194e-01, -1.1182e-01, -4.0494e-01, -2.9093e-01],
        [ 4.9670e-01,  2.1550e-01,  1.2293e-01, -3.2118e-02],
        [ 1.7454e-01,  1.9732e-01, -1.4001e-01,  1.9199e-01],
        [ 1.4528e-01, -3.9789e-02,  1.0010e-01, -3.7776e-01],
        [ 1.3963e-02, -4.4644e-01,  4.2921e-02,  4.6774e-01]])

In [70]:
# 参数绑定
shared = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))
net(x)
net[2] == net[4]

True

In [71]:
net[2].weight.data == net[4].weight.data

tensor([[True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True]])

In [73]:
net[2].bias.data[0] = 3

In [76]:
net[2].bias.data[0] == net[4].bias.data[0]
# 运算包含绑定层时，梯度叠加

tensor(True)